In [ ]:
#https://theeffectbook.net/ch-EventStudies.html?panelset1=python-code2&panelset=python-code

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt
from causaldata import google_stock
goog = google_stock.load_pandas().data
plt.figure(figsize=(20,10))
# Create estimation data set
goog['Date'] = pd.to_datetime(goog['Date'])
est_data = goog.loc[(goog['Date'] >= pd.Timestamp(2015,5,1)) & 
(goog['Date'] < pd.Timestamp(2015,7,31))] 

# And observation data
obs_data = goog.loc[(goog['Date'] >= pd.Timestamp(2015,8,6)) & 
(goog['Date'] < pd.Timestamp(2015,8,24))] 

# Estimate a model predicting stock price with market return
m = smf.ols('Google_Return ~ SP500_Return', data = est_data).fit()

# Get AR
# Using mean of estimation return
goog_return = np.mean(est_data['Google_Return'])
obs_data['AR_mean'] = obs_data['Google_Return'] - goog_return
# Then comparing to market return
obs_data['AR_market'] = obs_data['Google_Return'] - obs_data['SP500_Return']
# Then using model fit with estimation data
obs_data['risk_pred'] = m.predict(obs_data)
obs_data['AR_risk'] = obs_data['Google_Return'] - obs_data['risk_pred']

# Graph the results
sns.lineplot(x = obs_data['Date'],y = obs_data['AR_risk'])
plt.axvline(pd.Timestamp(2015,8,10))
plt.axhline(0)


In [ ]:
smf.ols('Google_Return ~ SP500_Return', data = est_data)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
np.random.seed(10)

# Ten groups with ten periods each
id = pd.DataFrame({'id': range(0,10), 'key': 1})
t = pd.DataFrame({'t': range(1,11), 'key': 1})
d = id.merge(t, on = 'key')
# Add an event in period 6 with a one-period effect
d['Y'] = np.random.normal(0,1,100) + 1*(d['t'] == 6)

# Estimate our model using time 5 as reference
m = smf.ols('Y~C(t, Treatment(reference = 5))', data = d)

# Fit with SEs clustered at the group level
m = m.fit(cov_type = 'cluster',cov_kwds={'groups': d['id']})

# Get coefficients and CIs 
# The original table will have an intercept up top
# But we'll overwrite it with our 5 reference
p = pd.DataFrame({'t': [5,1,2,3,4,6,7,8,9,10],
    'b': m.params, 'se': m.bse})
# And add our period-5 zero
p.iloc[0] = [5, 0, 0]

# Sort for plotting
p = p.sort_values('t')
# and make CIs by scaling the standard error
p['ci'] = 1.96*p['se']

# Plot the estimates as connected lines with error bars
plt.errorbar(x = 't', y = 'b',
 yerr = 'ci', data = p)
# Add a horizontal line at 0
plt.axhline(0, linestyle = 'dashed')
# And a vertical line at the treatment time
plt.axvline(5)

In [ ]:
b

# DID

In [ ]:
import linearmodels as lm
from causaldata import organ_donations
od = organ_donations.load_pandas().data

# Create Treatment Variable
od['California'] = od['State'] == 'California'
od['After'] = od['Quarter_Num'] > 3
od['Treated'] = 1*(od['California'] & od['After'])
# Set our individual and time (index) for our data
od = od.set_index(['State','Quarter_Num'])

mod = lm.PanelOLS.from_formula('''Rate ~ 
Treated + EntityEffects + TimeEffects''',od)

# Specify clustering when we fit the model
clfe = mod.fit(cov_type = 'clustered',
cluster_entity = True)
print(clfe)

In [ ]:
res

In [ ]:
import pandas as pd
import matplotlib as plt
import linearmodels as lm
from causaldata import organ_donations
od = organ_donations.load_pandas().data

# Create Treatment Variable
od['California'] = od['State'] == 'California'

# Create our interactions by hand,
# skipping quarter 3, the last one before treatment
for i in [1, 2, 4, 5, 6]:
    name = 'INX'+str(i)
    od[name] = 1*od['California']
    od.loc[od['Quarter_Num'] != i, name] = 0

# Set our individual and time (index) for our data
od = od.set_index(['State','Quarter_Num'])

mod = lm.PanelOLS.from_formula('''Rate ~ 
INX1 + INX2 + INX4 + INX5 + INX6 + 
EntityEffects + TimeEffects''',od)

# Specify clustering when we fit the model
clfe = mod.fit(cov_type = 'clustered',
cluster_entity = True)

# Get coefficients and CIs
res = pd.concat([clfe.params, clfe.std_errors], axis = 1)
# Scale standard error to CI
res['ci'] = res['std_error']*1.96

# Add our quarter values
res['Quarter_Num'] = [1, 2, 4, 5, 6]
# And add our reference period back in
reference = pd.DataFrame([[0,0,0,3]],
columns = ['parameter',
            'lower',
            'upper',
            'Quarter_Num'])
res = pd.concat([res, reference])

# For plotting, sort and add labels
res = res.sort_values('Quarter_Num')
res['Quarter'] = ['Q42010','Q12011',
                    'Q22011','Q32011',
                    'Q42011','Q12012']

# Plot the estimates as connected lines with error bars

plt.pyplot.errorbar(x = 'Quarter', y = 'parameter',
                    yerr = 'ci', data = res)
# Add a horizontal line at 0
plt.pyplot.axhline(0, linestyle = 'dashed')

In [ ]:
res